# Customer Segmentation and Predictive Analytics Using Machine Learning

**Assignment 6 | R Programming (ODD 2026-27)**

Purvaj Gaonkar | 23102C0083 | BE CMPN-C

---

## Problem Statement

An e-commerce company wants to better understand its customers and improve its
marketing decisions. Using historical online retail transaction data, this notebook
builds a machine-learning analytics solution that identifies meaningful customer
segments and predicts whether a customer belongs to a high-value category.

## Dataset

UCI Machine Learning Repository, **Online Retail** dataset: 541,909 transactions from
a UK-based online retailer between December 2010 and December 2011. Attributes are
InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID and
Country.

## Workflow

1. Preprocessing and customer-level feature engineering (RFM and extensions)
2. Elbow method for choosing k
3. K-Means segmentation
4. Hierarchical clustering and dendrogram
5. Silhouette evaluation and comparison
6. PCA projection to 2D
7. Cluster-wise profiling
8. Random Forest and SVM for high-value prediction
9. Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix
10. Feature importance
11. Interactive 3D Plotly visualisation
12. Segment-wise marketing recommendations

Runtime: **Python 3** (the Colab default). Run all.

---
## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, ConfusionMatrixDisplay)
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

pio.renderers.default = "colab"
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded.")

### Load the dataset

The file is an Excel workbook of roughly 23 MB, so the download takes a moment. It is
cached locally so a re-run does not fetch it again.

In [ ]:
import os, urllib.request

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
PATH = "online_retail.xlsx"

if not os.path.exists(PATH):
    print("Downloading...")
    urllib.request.urlretrieve(URL, PATH)
    print("Done.")
else:
    print("Using cached file.")

df = pd.read_excel(PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

---
## 2. Preprocessing

The raw data has three problems that must be handled before customer-level features
can be built, and each requires a decision rather than a default.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Rows with no CustomerID:", df["CustomerID"].isnull().sum(),
      f"({df['CustomerID'].isnull().mean():.1%})")

**Missing CustomerID.** About a quarter of rows carry no customer identifier. These
are guest checkouts. Since the entire analysis is customer-level, a row without an
identifier cannot be attributed to anyone and must be dropped here. Note this is a
real loss of revenue information, not a free operation.

**Cancellations.** Invoice numbers beginning with `C` are returns and carry negative
quantities. They are removed for segmentation, because a return is not a purchase
event and would distort recency and frequency. The count is reported so the decision
is visible.

**Invalid quantities and prices.** Zero or negative values outside cancellations are
data-entry errors and are removed.

In [ ]:
log = []
def step(name, before, after):
    log.append({"Step": name, "Rows before": before, "Rows after": after,
                "Removed": before - after})

n0 = len(df)

# Cancellations
df["IsCancellation"] = df["InvoiceNo"].astype(str).str.startswith("C")
n_cancel = df["IsCancellation"].sum()
df = df[~df["IsCancellation"]].copy()
step("Remove cancellations", n0, len(df))

# Missing CustomerID
n1 = len(df)
df = df.dropna(subset=["CustomerID"]).copy()
step("Drop missing CustomerID", n1, len(df))

# Invalid quantity / price
n2 = len(df)
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
step("Remove invalid qty/price", n2, len(df))

# Duplicates
n3 = len(df)
df = df.drop_duplicates()
step("Remove duplicates", n3, len(df))

df["CustomerID"] = df["CustomerID"].astype(int)
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

cleaning_log = pd.DataFrame(log)
print(f"Cancellation rows found: {n_cancel:,}")
cleaning_log

In [ ]:
print(f"Final transactions : {len(df):,}")
print(f"Unique customers   : {df['CustomerID'].nunique():,}")
print(f"Date range         : {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")
print(f"Total revenue      : {df['Revenue'].sum():,.2f}")

---
## 3. Customer-Level Feature Engineering

Transactions are aggregated to one row per customer. RFM is the classical basis:

- **Recency** — days since the customer's last purchase. Lower is better.
- **Frequency** — number of distinct invoices.
- **Monetary** — total revenue.

Three further features are added because RFM alone cannot distinguish a customer who
spends a lot across many small orders from one who spends the same in a single large
order.

In [ ]:
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print("Snapshot date (reference for recency):", snapshot.date())

first_purchase = df.groupby("CustomerID")["InvoiceDate"].min().rename("FirstPurchase")

rfm = df.groupby("CustomerID").agg(
    Recency        = ("InvoiceDate", lambda s: (snapshot - s.max()).days),
    Frequency      = ("InvoiceNo",   "nunique"),
    Monetary       = ("Revenue",     "sum"),
    TotalQuantity  = ("Quantity",    "sum"),
    UniqueProducts = ("StockCode",   "nunique"),
).reset_index()

rfm = rfm.merge(first_purchase, on="CustomerID")
rfm["AvgOrderValue"] = rfm["Monetary"] / rfm["Frequency"]
rfm["Tenure"] = (snapshot - rfm["FirstPurchase"]).dt.days
rfm["PurchaseRate"] = rfm["Frequency"] / (rfm["Tenure"] / 30.0).clip(lower=1)
rfm = rfm.drop(columns=["FirstPurchase"])

print("Customers:", len(rfm))
rfm.head()

In [ ]:
rfm.describe().T

### Distributions before treatment

RFM variables are always heavily right-skewed: most customers buy once and spend
little, a few spend enormously. This matters because K-Means minimises squared
Euclidean distance and is therefore dominated by extreme values.

In [ ]:
feat_cols = ["Recency", "Frequency", "Monetary", "TotalQuantity",
             "UniqueProducts", "AvgOrderValue", "PurchaseRate"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), feat_cols):
    sns.histplot(rfm[c], bins=50, ax=ax, color="steelblue")
    ax.set_title(c)
    ax.set_ylabel("")
axes.ravel()[-1].axis("off")
plt.suptitle("Feature distributions before treatment", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

print("Skewness:")
print(rfm[feat_cols].skew().round(2))

### Outlier treatment and scaling

Two decisions, both consequential.

**Winsorising at the 1st and 99th percentiles** caps the most extreme values instead
of deleting those customers. Deleting them would remove exactly the high-value
customers the second half of this assignment is meant to predict, which would defeat
the purpose.

**A log transform** then compresses the remaining skew so that the clustering is
driven by order-of-magnitude differences rather than by a handful of outliers. Only
after that is standardisation applied, since K-Means is scale-sensitive and Monetary
would otherwise dominate Recency purely because of its larger numeric range.

In [ ]:
X = rfm[feat_cols].copy()

# Winsorise at 1st / 99th percentile
for c in feat_cols:
    lo, hi = X[c].quantile([0.01, 0.99])
    X[c] = X[c].clip(lo, hi)

# Log1p to compress skew
X_log = np.log1p(X)

# Standardise
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)
X_scaled = pd.DataFrame(X_scaled, columns=feat_cols, index=rfm.index)

print("Skewness after log transform:")
print(X_log.skew().round(2))
X_scaled.describe().T[["mean", "std", "min", "max"]].round(3)

---
## 4. Elbow Method

Inertia is the within-cluster sum of squares. It falls as k rises, so the aim is not
the minimum but the point where the marginal gain drops sharply.

In [ ]:
inertias, sils = [], []
K_RANGE = range(2, 11)

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(list(K_RANGE), inertias, "o-", color="steelblue")
ax[0].set_xlabel("Number of clusters (k)"); ax[0].set_ylabel("Inertia (WCSS)")
ax[0].set_title("Elbow Method")
ax[1].plot(list(K_RANGE), sils, "o-", color="darkorange")
ax[1].set_xlabel("Number of clusters (k)"); ax[1].set_ylabel("Silhouette score")
ax[1].set_title("Silhouette score by k")
plt.tight_layout(); plt.show()

pd.DataFrame({"k": list(K_RANGE),
              "Inertia": np.round(inertias, 1),
              "Silhouette": np.round(sils, 4)})

In [ ]:
# Percentage drop in inertia at each step, to make the elbow explicit
drops = [np.nan] + [100 * (inertias[i-1] - inertias[i]) / inertias[i-1]
                    for i in range(1, len(inertias))]
pd.DataFrame({"k": list(K_RANGE), "Inertia": np.round(inertias, 1),
              "% drop from previous k": np.round(drops, 2)})

The elbow and the silhouette peak need not agree, and when they disagree the choice
has to be argued rather than asserted. **k = 4** is used below: it sits at or near the
elbow, gives a competitive silhouette, and produces segments that map onto
recognisable commercial categories. A k chosen purely to maximise silhouette often
yields two clusters, which is statistically clean but useless for marketing.

In [ ]:
K_OPT = 4
kmeans = KMeans(n_clusters=K_OPT, random_state=RANDOM_STATE, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(X_scaled)

print("Cluster sizes:")
print(rfm["Cluster"].value_counts().sort_index())
print(f"\nSilhouette score (k={K_OPT}): {silhouette_score(X_scaled, rfm['Cluster']):.4f}")

---
## 5. Hierarchical Clustering and Dendrogram

Agglomerative clustering starts with every customer as its own cluster and merges the
closest pair repeatedly. Ward linkage is used because it minimises the increase in
within-cluster variance at each merge, which is the same objective K-Means pursues,
making the two directly comparable.

In [ ]:
linked = linkage(X_scaled, method="ward")

plt.figure(figsize=(14, 6))
dendrogram(linked, truncate_mode="lastp", p=30,
           leaf_rotation=90, leaf_font_size=10, show_contracted=True)
plt.title("Hierarchical Clustering Dendrogram (Ward linkage, last 30 merges)")
plt.xlabel("Cluster size or sample index")
plt.ylabel("Ward distance")
plt.axhline(y=linked[-(K_OPT-1), 2], color="red", linestyle="--",
            label=f"Cut for {K_OPT} clusters")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
rfm["HCluster"] = fcluster(linked, K_OPT, criterion="maxclust") - 1

print("Hierarchical cluster sizes:")
print(rfm["HCluster"].value_counts().sort_index())
print(f"\nSilhouette (hierarchical): {silhouette_score(X_scaled, rfm['HCluster']):.4f}")

### Comparing the two methods

In [ ]:
comparison = pd.DataFrame({
    "Method": ["K-Means", "Hierarchical (Ward)"],
    "Clusters": [K_OPT, K_OPT],
    "Silhouette": [silhouette_score(X_scaled, rfm["Cluster"]),
                   silhouette_score(X_scaled, rfm["HCluster"])],
    "Largest cluster %": [100*rfm["Cluster"].value_counts(normalize=True).max(),
                          100*rfm["HCluster"].value_counts(normalize=True).max()],
}).round(4)
comparison

In [ ]:
ct = pd.crosstab(rfm["Cluster"], rfm["HCluster"],
                 rownames=["K-Means"], colnames=["Hierarchical"])
plt.figure(figsize=(7, 5))
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues")
plt.title("Agreement between K-Means and Hierarchical assignments")
plt.tight_layout(); plt.show()
ct

A strong diagonal means the two methods largely agree, which is reassuring: the
structure is a property of the data rather than an artefact of one algorithm. Cells
off the diagonal mark customers sitting near a boundary, where the two methods split
differently.

### Silhouette plot

The aggregate silhouette hides how the score is distributed. A cluster whose members
mostly score near zero is poorly separated even if the overall average looks
acceptable, and negative values mark customers who would fit a different cluster
better.

In [ ]:
sil_vals = silhouette_samples(X_scaled, rfm["Cluster"])
avg = silhouette_score(X_scaled, rfm["Cluster"])

fig, ax = plt.subplots(figsize=(9, 6))
y_lower = 10
for i in range(K_OPT):
    vals = np.sort(sil_vals[rfm["Cluster"] == i])
    y_upper = y_lower + len(vals)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, alpha=0.75)
    ax.text(-0.05, y_lower + 0.5*len(vals), str(i))
    y_lower = y_upper + 10
ax.axvline(avg, color="red", linestyle="--", label=f"Mean = {avg:.3f}")
ax.set_xlabel("Silhouette coefficient"); ax.set_ylabel("Cluster")
ax.set_title("Silhouette plot for K-Means clusters")
ax.legend(); plt.tight_layout(); plt.show()

print("Customers with negative silhouette:", (sil_vals < 0).sum(),
      f"({(sil_vals < 0).mean():.1%})")

---
## 6. PCA Projection

In [ ]:
pca = PCA(n_components=len(feat_cols), random_state=RANDOM_STATE)
pcs = pca.fit_transform(X_scaled)

evr = pca.explained_variance_ratio_
pd.DataFrame({
    "Component": [f"PC{i+1}" for i in range(len(evr))],
    "Explained variance": np.round(evr, 4),
    "Cumulative": np.round(np.cumsum(evr), 4)
})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].bar(range(1, len(evr)+1), evr, color="steelblue")
ax[0].set_xlabel("Principal component"); ax[0].set_ylabel("Explained variance ratio")
ax[0].set_title("Scree plot")
ax[1].plot(range(1, len(evr)+1), np.cumsum(evr), "o-", color="darkorange")
ax[1].axhline(0.8, color="red", linestyle="--", label="80%")
ax[1].set_xlabel("Number of components"); ax[1].set_ylabel("Cumulative variance")
ax[1].set_title("Cumulative explained variance"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
rfm["PC1"], rfm["PC2"], rfm["PC3"] = pcs[:, 0], pcs[:, 1], pcs[:, 2]

plt.figure(figsize=(10, 7))
sns.scatterplot(data=rfm, x="PC1", y="PC2", hue="Cluster",
                palette="Set2", s=28, alpha=0.75)
plt.title(f"Customer segments in PCA space "
          f"(PC1 + PC2 = {evr[0]+evr[1]:.1%} of variance)")
plt.xlabel(f"PC1 ({evr[0]:.1%})"); plt.ylabel(f"PC2 ({evr[1]:.1%})")
plt.legend(title="Cluster"); plt.tight_layout(); plt.show()

### What the components mean

A principal component is a weighted combination of the original features. Reading the
loadings turns an abstract axis into something interpretable.

In [ ]:
loadings = pd.DataFrame(pca.components_[:3].T,
                        columns=["PC1", "PC2", "PC3"], index=feat_cols)
plt.figure(figsize=(7, 5))
sns.heatmap(loadings, annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("PCA loadings: contribution of each feature")
plt.tight_layout(); plt.show()
loadings.round(3)

---
## 7. Cluster Profiling

In [ ]:
profile = rfm.groupby("Cluster")[feat_cols].mean().round(2)
profile["Customers"] = rfm["Cluster"].value_counts().sort_index()
profile["% of base"] = (100 * profile["Customers"] / len(rfm)).round(1)
profile["Total revenue"] = rfm.groupby("Cluster")["Monetary"].sum().round(0)
profile["% of revenue"] = (100 * profile["Total revenue"] / rfm["Monetary"].sum()).round(1)
profile

In [ ]:
# Normalised profile so clusters can be compared on one scale
norm = rfm.groupby("Cluster")[feat_cols].mean()
norm = (norm - norm.min()) / (norm.max() - norm.min())

plt.figure(figsize=(10, 5))
sns.heatmap(norm, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Cluster profiles (each feature scaled 0-1 across clusters)")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, c in zip(axes, ["Recency", "Frequency", "Monetary"]):
    sns.boxplot(data=rfm, x="Cluster", y=c, ax=ax, palette="Set2", showfliers=False)
    ax.set_title(f"{c} by cluster")
plt.tight_layout(); plt.show()

In [ ]:
# Automatic labelling based on where each cluster sits on R, F and M
med = rfm[["Recency", "Frequency", "Monetary"]].median()
labels = {}
for c in sorted(rfm["Cluster"].unique()):
    s = rfm[rfm["Cluster"] == c][["Recency", "Frequency", "Monetary"]].mean()
    if s["Monetary"] >= rfm.groupby("Cluster")["Monetary"].mean().max():
        labels[c] = "Champions / High Value"
    elif s["Recency"] > med["Recency"] * 1.5:
        labels[c] = "At Risk / Lapsed"
    elif s["Frequency"] >= med["Frequency"]:
        labels[c] = "Loyal Regulars"
    else:
        labels[c] = "New / Occasional"

rfm["Segment"] = rfm["Cluster"].map(labels)
print("Segment labels:")
for k, v in labels.items():
    print(f"  Cluster {k}: {v}")
rfm["Segment"].value_counts()

---
## 8. Defining the High-Value Target

The cluster with the highest mean monetary value becomes the positive class. Two
consequences follow and both matter for how the results should be read.

First, the label is **derived from the clustering**, so the classifiers are learning to
reproduce a boundary that K-Means drew using the same features. High accuracy is
therefore expected and is not evidence of predicting anything external.

Second, the classes are **imbalanced**, since high-value customers are a minority by
construction. Accuracy alone would be misleading, which is why precision, recall, F1
and ROC-AUC are all reported.

In [ ]:
high_value_cluster = rfm.groupby("Cluster")["Monetary"].mean().idxmax()
rfm["HighValue"] = (rfm["Cluster"] == high_value_cluster).astype(int)

print(f"High-value cluster: {high_value_cluster} ({labels[high_value_cluster]})")
print(f"High-value customers: {rfm['HighValue'].sum():,} "
      f"({rfm['HighValue'].mean():.1%} of base)")
print(f"Revenue share       : "
      f"{100*rfm.loc[rfm['HighValue']==1,'Monetary'].sum()/rfm['Monetary'].sum():.1f}%")

plt.figure(figsize=(6, 4))
rfm["HighValue"].value_counts().plot(kind="bar", color=["steelblue", "darkorange"])
plt.xticks([0, 1], ["Standard", "High Value"], rotation=0)
plt.ylabel("Customers"); plt.title("Class balance")
plt.tight_layout(); plt.show()

### Train / test split

`stratify` preserves the class ratio in both splits. Without it a rare positive class
can end up badly represented in the test set purely by chance.

In [ ]:
FEATURES = feat_cols
X_ml = rfm[FEATURES]
y_ml = rfm["HighValue"]

X_train, X_test, y_train, y_test = train_test_split(
    X_ml, y_ml, test_size=0.25, random_state=RANDOM_STATE, stratify=y_ml)

print(f"Train: {X_train.shape[0]} rows, positive rate {y_train.mean():.1%}")
print(f"Test : {X_test.shape[0]} rows, positive rate {y_test.mean():.1%}")

---
## 9. Model 1 — Random Forest

`class_weight="balanced"` makes the model weight the minority class in inverse
proportion to its frequency, so it cannot score well by simply predicting the majority
everywhere.

In [ ]:
rf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=None,
                                   class_weight="balanced",
                                   random_state=RANDOM_STATE, n_jobs=-1))
])
rf.fit(X_train, y_train)

rf_pred  = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]
print("Random Forest trained.")
print(classification_report(y_test, rf_pred,
                            target_names=["Standard", "High Value"]))

## 10. Model 2 — Support Vector Machine

An RBF kernel SVM. Scaling is essential here in a way it is not for a tree: the RBF
kernel is a function of Euclidean distance, so an unscaled Monetary column would
swamp every other feature. `probability=True` enables the probability estimates that
ROC-AUC needs.

In [ ]:
svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", C=1.0, gamma="scale",
                class_weight="balanced", probability=True,
                random_state=RANDOM_STATE))
])
svm.fit(X_train, y_train)

svm_pred  = svm.predict(X_test)
svm_proba = svm.predict_proba(X_test)[:, 1]
print("SVM trained.")
print(classification_report(y_test, svm_pred,
                            target_names=["Standard", "High Value"]))

---
## 11. Model Comparison

In [ ]:
def evaluate(name, y_true, y_pred, y_proba):
    return {
        "Model":     name,
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_true, y_proba),
    }

results = pd.DataFrame([
    evaluate("Random Forest", y_test, rf_pred,  rf_proba),
    evaluate("SVM (RBF)",     y_test, svm_pred, svm_proba),
]).set_index("Model").round(4)
results

In [ ]:
ax = results.plot(kind="bar", figsize=(11, 5), width=0.8, colormap="Set2")
ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
ax.set_title("Random Forest vs SVM across all metrics")
ax.legend(loc="lower right", ncol=5, fontsize=9)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### Confusion matrices

The off-diagonal cells are what matter commercially. A false negative is a high-value
customer the model failed to flag, so they miss out on retention effort. A false
positive is a standard customer receiving premium treatment, which costs money but
does no lasting harm. Those two errors are not equally expensive, which is why recall
deserves more weight than precision here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, pred) in zip(axes, [("Random Forest", rf_pred), ("SVM (RBF)", svm_pred)]):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=["Standard", "High Value"]).plot(
        ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}")
    ax.grid(False)
plt.tight_layout(); plt.show()

for name, pred in [("Random Forest", rf_pred), ("SVM", svm_pred)]:
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"{name:15s}  TN={tn:4d}  FP={fp:3d}  FN={fn:3d}  TP={tp:3d}"
          f"   (missed high-value customers: {fn})")

In [ ]:
plt.figure(figsize=(7.5, 6))
for name, proba in [("Random Forest", rf_proba), ("SVM (RBF)", svm_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc_score(y_test, proba):.4f})")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC = 0.5)")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves"); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

### Cross-validation

A single split can flatter a model by chance. Stratified 5-fold cross-validation
reports the spread as well as the mean.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, model in [("Random Forest", rf), ("SVM (RBF)", svm)]:
    scores = cross_val_score(model, X_ml, y_ml, cv=cv, scoring="f1", n_jobs=-1)
    print(f"{name:15s} F1: {scores.mean():.4f} +/- {scores.std():.4f}   {np.round(scores,4)}")

---
## 12. Feature Importance

In [ ]:
importances = pd.Series(rf.named_steps["clf"].feature_importances_,
                        index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.xlabel("Gini importance"); plt.title("Random Forest feature importance")
plt.tight_layout(); plt.show()
importances.round(4)

Gini importance is biased toward high-cardinality features. Permutation importance
measures the actual drop in performance when a feature is shuffled, which is a more
honest estimate and is computed on held-out data.

In [ ]:
perm = permutation_importance(rf, X_test, y_test, n_repeats=10,
                              random_state=RANDOM_STATE, n_jobs=-1, scoring="f1")
perm_s = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=perm_s.values, y=perm_s.index, palette="magma")
plt.xlabel("Mean F1 drop when shuffled")
plt.title("Permutation importance (test set)")
plt.tight_layout(); plt.show()

pd.DataFrame({"Gini": importances, "Permutation": perm_s}).round(4)

---
## 13. Interactive 3D Visualisation

In [ ]:
plot_df = rfm.copy()
plot_df["Cluster"] = plot_df["Cluster"].astype(str)

fig = px.scatter_3d(
    plot_df, x="PC1", y="PC2", z="PC3",
    color="Segment", opacity=0.75,
    hover_data={"CustomerID": True, "Recency": True,
                "Frequency": True, "Monetary": ":.0f",
                "PC1": False, "PC2": False, "PC3": False},
    title="Customer segments in 3D PCA space",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(marker=dict(size=3.5))
fig.update_layout(height=700, legend_title_text="Segment")
fig.show()

In [ ]:
fig = px.scatter_3d(
    plot_df, x="Recency", y="Frequency", z="Monetary",
    color="Segment", opacity=0.75, log_z=True,
    hover_data={"CustomerID": True},
    title="Customer segments in raw RFM space (log Monetary)",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(marker=dict(size=3.5))
fig.update_layout(height=700)
fig.show()

---
## 14. Marketing Recommendations

Each recommendation is tied to the numbers in the profile table rather than stated
generically.

In [ ]:
rec_rows = []
for c in sorted(rfm["Cluster"].unique()):
    sub = rfm[rfm["Cluster"] == c]
    rec_rows.append({
        "Cluster": c,
        "Segment": labels[c],
        "Customers": len(sub),
        "% of base": round(100*len(sub)/len(rfm), 1),
        "% of revenue": round(100*sub["Monetary"].sum()/rfm["Monetary"].sum(), 1),
        "Avg Recency (d)": round(sub["Recency"].mean()),
        "Avg Frequency": round(sub["Frequency"].mean(), 1),
        "Avg Monetary": round(sub["Monetary"].mean()),
    })
rec = pd.DataFrame(rec_rows)
rec

**Champions / High Value.** Small in number but responsible for a disproportionate
share of revenue, which makes them the highest-risk group to lose. Priority is
retention rather than acquisition: dedicated account contact, early access to stock,
loyalty tiers. Spend here is justified by the revenue concentration shown in the table
above, not by headcount.

**Loyal Regulars.** Buy often, spend moderately. The lever is basket size rather than
visit count, so bundling, volume thresholds for free shipping, and recommendations
based on products they already buy. Their frequency indicates the relationship already
works; the aim is to increase value per visit.

**At Risk / Lapsed.** Recency is well above the median, meaning they have stopped
buying without formally leaving. This group deserves the earliest intervention,
because reactivation is cheaper than acquisition and the window narrows with time.
Win-back offers and a survey to identify why they stopped.

**New / Occasional.** Low frequency and low monetary, but this group also contains
genuinely new customers who have simply not had time to accumulate history. Tenure
should be checked before writing them off. Onboarding sequences and a second-purchase
incentive, since the step from one purchase to two is the strongest single predictor
of retention.

### A caution on the classifier

The high-value label was derived from the clustering, using the same features the
models were then given. The classifiers are therefore learning a boundary that K-Means
had already drawn, and strong metrics confirm the segments are cleanly separable
rather than demonstrating predictive power on anything external. A genuine test would
be to train on one period and predict who becomes high-value in the next, which the
current setup does not do.

---
## 15. Conclusion

In [ ]:
print("=" * 62)
print("SUMMARY")
print("=" * 62)
print(f"Transactions after cleaning : {len(df):,}")
print(f"Customers segmented         : {len(rfm):,}")
print(f"Clusters (k)                : {K_OPT}")
print(f"Silhouette, K-Means         : {silhouette_score(X_scaled, rfm['Cluster']):.4f}")
print(f"Silhouette, Hierarchical    : {silhouette_score(X_scaled, rfm['HCluster']):.4f}")
print(f"PC1+PC2 variance explained  : {evr[0]+evr[1]:.1%}")
print(f"High-value cluster          : {high_value_cluster} ({labels[high_value_cluster]})")
print(f"High-value share of base    : {rfm['HighValue'].mean():.1%}")
print(f"High-value share of revenue : "
      f"{100*rfm.loc[rfm['HighValue']==1,'Monetary'].sum()/rfm['Monetary'].sum():.1f}%")
print("-" * 62)
print(results.to_string())
print("-" * 62)
print(f"Most important feature      : {perm_s.index[0]}")
print("=" * 62)

In [ ]:
rfm.to_csv("customer_segments.csv", index=False)
profile.to_csv("cluster_profiles.csv")
results.to_csv("model_comparison.csv")
print("Saved: customer_segments.csv, cluster_profiles.csv, model_comparison.csv")

**Findings.**

Segmentation is stable across methods. K-Means and Ward hierarchical clustering
produce similar partitions with comparable silhouette scores, and the crosstab shows
most customers assigned identically by both. That agreement suggests the structure is
in the data rather than imposed by one algorithm.

Revenue is concentrated. The high-value segment is a small fraction of the customer
base but accounts for a share of revenue far exceeding its headcount, which is the
single most actionable finding and the reason retention spend should be skewed toward
it.

Both classifiers separate the segment well, with performance close enough that the
choice between them rests on other grounds: Random Forest gives interpretable feature
importances and trains faster, while SVM offers no comparable built-in explanation.
For a task where marketing teams need to understand why a customer was flagged,
Random Forest is the more practical choice.

**Limitations.** The analysis covers a single year from one UK-based retailer, so the
segments are specific to that period and market. Removing rows without a CustomerID
discarded around a quarter of transactions, and those guest purchases carry real
revenue that is absent from every figure here. Finally, as noted above, the supervised
step validates cluster separability rather than forecasting future value.